In [1]:
import asyncio
from typing import List
from benchmarks import benchmark_orchestrator
from benchmarks.data_models import BenchmarkRunResult
from benchmarks.benchmark_candidates import CANDIDATE_GENERATORS
import pandas as pd
from benchmarks.logger import JsonTraceLogger


# Set pandas display options
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

App name mismatch detected. The runner is configured with app name "InMemoryRunner", but the root agent was loaded from "/Users/ivanmkc/Documents/code/adk-python/src/google/adk/agents", which implies app name "agents".


App name mismatch detected. The runner is configured with app name "InMemoryRunner", but the root agent was loaded from "/Users/ivanmkc/Documents/code/adk-python/src/google/adk/agents", which implies app name "agents".


In [2]:
# ANSI escape codes for colors
class bcolors:
    HEADER = '\033[95m'
    OKBLUE = '\033[94m'
    OKCYAN = '\033[96m'
    OKGREEN = '\033[92m'
    WARNING = '\033[93m'
    FAIL = '\033[91m'
    ENDC = '\033[0m'
    BOLD = '\033[1m'
    UNDERLINE = '\033[4m'

logger = JsonTraceLogger(output_dir="traces")

JSON trace log will be written to traces/trace_2025-11-26_22-25-10.jsonl


In [3]:
async def run_comparison() -> List[BenchmarkRunResult]:
    """Sets up and runs the benchmark comparison."""
    print("Configuring benchmark run...")
    
    benchmark_suites = [
        "benchmarks/benchmark_definitions/api_understanding/benchmark.yaml",
        "benchmarks/benchmark_definitions/fix_errors/benchmark.yaml",
        "benchmarks/benchmark_definitions/diagnose_setup_errors_mc/benchmark.yaml",
        "benchmarks/benchmark_definitions/configure_adk_features_mc/benchmark.yaml",
        "benchmarks/benchmark_definitions/predict_runtime_behavior_mc/benchmark.yaml",
    ]
    
    answer_generators = CANDIDATE_GENERATORS
    
    print("Executing benchmarks...")
    results = await benchmark_orchestrator.run_benchmarks(
        benchmark_suites=benchmark_suites, 
        answer_generators=answer_generators,
        max_concurrency=20,
        logger=logger,
    )
    
    return results

def extract_error_type(row) -> str:
    """Extracts error type from the result row."""
    if "error_type" in row and pd.notna(row["error_type"]):
        # If it's an Enum object (from pydantic validation), get its value
        et = row["error_type"]
        if hasattr(et, "value"):
            return et.value
        return str(et)
    return "OtherError"

In [4]:
async def main():
    # Execute the benchmarks
    results = await run_comparison()
    raw_results_df = pd.DataFrame([r.model_dump() for r in results])
    
    if not raw_results_df.empty:
        raw_results_df["suite"] = raw_results_df["suite"].apply(lambda x: x.split("/")[-2])
        raw_results_df["final_error_type"] = raw_results_df.apply(extract_error_type, axis=1)

        # 1. General Pass/Total Summary
        summary_df = (
            raw_results_df.groupby(["answer_generator", "suite"])
            .agg(
                passed=("result", "sum"),
                total=("result", "count"),
            )
        )
        summary_df["pass_rate"] = summary_df["passed"] / summary_df["total"]

        print(f"{bcolors.HEADER}--- Benchmark Summary ---{bcolors.ENDC}")
        print(summary_df)
        print("\n")

        # 2. Detailed Error Breakdown with Ratios
        # Filter for failures only
        failed_df = raw_results_df[raw_results_df["result"] == 0]
        
        if not failed_df.empty:
            # Calculate counts per error type
            error_counts = (
                failed_df.groupby(["answer_generator", "suite", "final_error_type"])
                .size()
                .reset_index(name="count")
            )
            
            # Merge with total counts to calculate ratios relative to total runs
            # First, get total counts per generator/suite group
            total_counts = raw_results_df.groupby(["answer_generator", "suite"]).size().reset_index(name="total_runs")
            
            # Merge error counts with totals
            error_summary = pd.merge(error_counts, total_counts, on=["answer_generator", "suite"])
            
            # Calculate failure rate for each specific error type
            error_summary["failure_ratio"] = error_summary["count"] / error_summary["total_runs"]
            
            print(f"{bcolors.HEADER}--- Detailed Error Breakdown ---{bcolors.ENDC}")
            # Sort for better readability
            error_summary = error_summary.sort_values(["answer_generator", "suite", "count"], ascending=[True, True, False])
            print(error_summary.to_string(index=False))

            # --- DETAILED DEBUG FOR GEMINI CLI FAILURES ---
            print(f"\n{bcolors.FAIL}--- DETAILED GEMINI CLI FAILURES ---{bcolors.ENDC}")
            cli_failures = failed_df[failed_df["answer_generator"].str.contains("GeminiCliAnswerGenerator")]
            if not cli_failures.empty:
                # Print just the first 3 failures to avoid overwhelming output
                for idx, row in cli_failures.head(3).iterrows():
                    print(f"\nBenchmark: {row['benchmark_name']} (Suite: {row['suite']})")
                    print(f"Error Type: {row['final_error_type']}")
                    print(f"Full Validation Error:\n{row['validation_error']}")
                    print("-" * 60)
            else:
                print("No Gemini CLI failures found in this run.")
            # -----------------------------------------------
        else:
            print(f"{bcolors.OKGREEN}No failures detected!{bcolors.ENDC}")
    else:
        print("No results returned.")

if __name__ == "__main__":
    await main()

Configuring benchmark run...
Executing benchmarks...
--- Loading benchmark suite: benchmarks/benchmark_definitions/api_understanding/benchmark.yaml ---
  - Queuing tests for answer generator: AdkAnswerGenerator(adk_test_agent)
  - Queuing tests for answer generator: AdkAnswerGenerator(adk_test_agent)
  - Queuing tests for answer generator: GeminiAnswerGenerator(gemini-2.5-flash)
  - Queuing tests for answer generator: GeminiAnswerGenerator(gemini-2.5-flash)-with-context-llms.txt
  - Queuing tests for answer generator: GeminiAnswerGenerator(gemini-2.5-pro)
  - Queuing tests for answer generator: GeminiAnswerGenerator(gemini-2.5-pro)-with-context-llms.txt
  - Queuing tests for answer generator: GeminiCliAnswerGenerator(gemini-2.5-flash)
  - Queuing tests for answer generator: GroundTruthAnswerGenerator
  - Queuing tests for answer generator: TrivialAnswerGenerator
--- Loading benchmark suite: benchmarks/benchmark_definitions/fix_errors/benchmark.yaml ---
  - Queuing tests for answer gene

  0%|                                                                 | 0/1773 [00:00<?, ?it/s]

  0%|▏                                                        | 5/1773 [00:03<22:34,  1.31it/s]

  0%|▏                                                        | 7/1773 [00:04<17:57,  1.64it/s]

  0%|▎                                                        | 8/1773 [00:04<16:20,  1.80it/s]

...

100%|█████████████████████████████████████████████████████▉| 1771/1773 [14:34<00:26, 13.13s/it]

100%|█████████████████████████████████████████████████████▉| 1772/1773 [15:17<00:22, 22.08s/it]

100%|██████████████████████████████████████████████████████| 1773/1773 [15:25<00:00, 17.82s/it]

100%|██████████████████████████████████████████████████████| 1773/1773 [15:25<00:00,  1.92it/s]

JSON trace log written to traces/trace_2025-11-26_22-25-10.jsonl
--- Benchmark Summary ---
                                                                                           passed  \
answer_generator                                              suite                                 
AdkAnswerGenerator(adk_test_agent)                            api_understanding                 0   
                                                              configure_adk_features_mc         1   
                                                              diagnose_setup_errors_mc          0   
                                                              fix_errors                        1   
                                                              predict_runtime_behavior_mc       0   
GeminiAnswerGenerator(gemini-2.5-flash)                       api_understanding                 9   
                                                              configure_adk_features_mc        70   
